# Name That LLM — offline submission

Generated from `configs/baseline.yaml`. This notebook uses only Kaggle's mounted Competition Data, installs/downloads nothing, and writes `submission.parquet` to the working directory.

Config version: **9**; source: `baseline.yaml`.


In [ ]:
# Generated by scripts/build_submission_notebook.py; edit baseline.yaml and regenerate.
from pathlib import Path
import hashlib
import json
import os
import tempfile
import time

import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

CONFIG_VERSION = 9
CONFIG_SHA256 = '39359217b1ecbf06e2c28b594f1f0360ad8ec782ce24389f116ffc0d02e23352'
MODEL_CANDIDATE = 'char_tfidf_svc'
MODEL_SETTINGS = {'analyzer': 'char',
 'c': 1.0,
 'max_features': 100000,
 'min_df': 2,
 'ngram_range': (3, 5),
 'sublinear_tf': True}
ALLOWED_LABELS = ['gpt-5.5',
 'gpt-5.6-luna',
 'gpt-5.6-sol',
 'gpt-5.6-terra',
 'gpt-6-astra',
 'gemini-3.1-pro-preview',
 'gemini-3.6-flash',
 'gemini-3.7-flash',
 'gemini-3.8-flash',
 'claude-fable-5',
 'claude-opus-4-6',
 'claude-opus-4-8',
 'claude-opus-5',
 'claude-sonnet-5',
 'qwen3.8-max',
 'glm-5.3',
 'glm-5.3-flash',
 'kimi-k3']
SEED = 42
ID_COLUMN = 'id'
GROUP_COLUMN = 'prompt_id'
TEXT_COLUMN = 'text'
LABEL_COLUMN = 'model'


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


competition_dir = Path("/kaggle/input/competitions/name-that-llm")
if os.environ.get("NTLLM_DATA_DIR"):
    data_dir = Path(os.environ["NTLLM_DATA_DIR"])
elif competition_dir.is_dir():
    data_dir = competition_dir
else:
    data_dir = Path("kaggle")
working_dir = Path(os.environ.get("NTLLM_OUTPUT_DIR", "/kaggle/working"))
if not os.environ.get("NTLLM_OUTPUT_DIR") and not working_dir.is_dir():
    working_dir = Path("reports/submissions")
train_path = data_dir / "train.csv"
test_path = data_dir / "test.csv"
submission_path = working_dir / "submission.parquet"
summary_path = working_dir / "run_summary.json"

if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError(f"expected train.csv and test.csv under {data_dir}")

train = pd.read_csv(train_path, dtype=str, keep_default_na=False)
test = pd.read_csv(test_path, dtype=str, keep_default_na=False)
required_train = {ID_COLUMN, GROUP_COLUMN, TEXT_COLUMN, LABEL_COLUMN}
required_test = {ID_COLUMN, TEXT_COLUMN}
if not required_train.issubset(train.columns):
    raise ValueError(f"train.csv is missing required columns: {sorted(required_train - set(train.columns))}")
if not required_test.issubset(test.columns):
    raise ValueError(f"test.csv is missing required columns: {sorted(required_test - set(test.columns))}")

for frame_name, frame, required in (("train", train, required_train), ("test", test, required_test)):
    for column in required:
        if frame[column].str.strip().eq("").any():
            raise ValueError(f"{frame_name}.csv has blank values in {column!r}")
    if frame[ID_COLUMN].duplicated().any():
        raise ValueError(f"{frame_name}.csv has duplicate IDs")

observed_labels = set(train[LABEL_COLUMN])
if observed_labels != set(ALLOWED_LABELS):
    raise ValueError("train labels do not exactly match the official allowlist")

vectorizer = TfidfVectorizer(
    analyzer=MODEL_SETTINGS["analyzer"],
    ngram_range=MODEL_SETTINGS["ngram_range"],
    min_df=MODEL_SETTINGS["min_df"],
    max_features=MODEL_SETTINGS["max_features"],
    sublinear_tf=MODEL_SETTINGS["sublinear_tf"],
)
model = Pipeline([
    ("tfidf", vectorizer),
    ("classifier", LinearSVC(C=MODEL_SETTINGS["c"], dual=True, random_state=SEED)),
])

fit_started = time.perf_counter()
model.fit(train[TEXT_COLUMN].tolist(), train[LABEL_COLUMN].tolist())
fit_seconds = time.perf_counter() - fit_started
predict_started = time.perf_counter()
predictions = model.predict(test[TEXT_COLUMN].tolist())
predict_seconds = time.perf_counter() - predict_started

if len(predictions) != len(test):
    raise ValueError("model returned a different number of predictions than test rows")
if not set(predictions).issubset(set(ALLOWED_LABELS)):
    raise ValueError("model predicted a label outside the official allowlist")
submission = pd.DataFrame({ID_COLUMN: test[ID_COLUMN].tolist(), "model": predictions})
if list(submission.columns) != ["id", "model"]:
    raise ValueError("submission columns must be exactly ['id', 'model']")
if submission["id"].isna().any() or submission["model"].isna().any():
    raise ValueError("submission contains null values")
if submission["id"].duplicated().any():
    raise ValueError("submission contains duplicate IDs")
if submission["id"].tolist() != test[ID_COLUMN].tolist():
    raise ValueError("submission IDs do not preserve test input order")

working_dir.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(prefix=".submission.", suffix=".parquet", dir=working_dir, delete=False) as stream:
    temporary_submission = Path(stream.name)
try:
    submission.to_parquet(temporary_submission, index=False)
    os.replace(temporary_submission, submission_path)
except Exception:
    temporary_submission.unlink(missing_ok=True)
    raise

summary = {
    "config_version": CONFIG_VERSION,
    "config_sha256": CONFIG_SHA256,
    "model_candidate": MODEL_CANDIDATE,
    "train_sha256": sha256_file(train_path),
    "test_sha256": sha256_file(test_path),
    "train_rows": len(train),
    "test_rows": len(test),
    "train_prompt_groups": int(train[GROUP_COLUMN].nunique()),
    "fit_seconds": fit_seconds,
    "predict_seconds": predict_seconds,
    "python_version": __import__("platform").python_version(),
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "submission_path": str(submission_path),
}
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2, sort_keys=True))
